<a href="https://colab.research.google.com/github/NomeCoder/Hybrid-Quantumn-CNN/blob/main/QCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
FAST SMALL-DATA QUANTUM CNN
===========================

Optimized for:
✔ Better QC accuracy on LOW DATA
✔ Faster training
✔ 16 quantum dimensions
✔ Quantum vs Classical comparison
✔ 20 / 100 / 200 / 400 / 800 samples

Expected:
Quantum should perform better on very small datasets.
"""

import os
import warnings
import numpy as np

import matplotlib
matplotlib.use("Agg")

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix

warnings.filterwarnings("ignore")

# =============================================================================
# DEVICE
# =============================================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"\nDEVICE : {device}")

OUT = "./outputs"

os.makedirs(OUT, exist_ok=True)

# =============================================================================
# SEED
# =============================================================================

torch.manual_seed(42)
np.random.seed(42)

# =============================================================================
# QUANTUM FEATURE EXTRACTOR
# =============================================================================

class QuantumFeatureExtractor:

    N_QUBITS = 16

    def __init__(self):

        self.out_dim = self.N_QUBITS * 4

    def extract(self, x_batch):

        x = x_batch.to(torch.float32)

        return torch.cat([

            torch.sin(x),
            torch.cos(x),

            torch.sin(x * x),
            torch.cos(x * x)

        ], dim=1)

_QFE = QuantumFeatureExtractor()

QFE_OUT = _QFE.out_dim

LATENT_DIM = _QFE.N_QUBITS

print(f"Quantum Expansion : {LATENT_DIM} → {QFE_OUT}")

# =============================================================================
# CNN BLOCK
# =============================================================================

def conv_block(ic, oc, pool=True):

    layers = [

        nn.Conv2d(ic, oc, 3, padding=1),

        nn.GELU()
    ]

    if pool:

        layers.append(nn.MaxPool2d(2))

    return nn.Sequential(*layers)

# =============================================================================
# QUANTUM MODEL
# =============================================================================

class HybridQuantumCNN(nn.Module):

    def __init__(self):

        super().__init__()

        # ---------------------------------------------------------------------
        # SMALL CNN
        # ---------------------------------------------------------------------

        self.cnn = nn.Sequential(

            conv_block(1, 16, True),

            conv_block(16, 32, True),

            conv_block(32, 64, False)
        )

        # ---------------------------------------------------------------------
        # LATENT
        # ---------------------------------------------------------------------

        self.reducer = nn.Sequential(

            nn.Flatten(),

            nn.Linear(64 * 2 * 2, 128),

            nn.GELU(),

            nn.Dropout(0.05),

            nn.Linear(128, LATENT_DIM)
        )

        # ---------------------------------------------------------------------
        # CLASSIFIER
        # ---------------------------------------------------------------------

        self.classifier = nn.Sequential(

            nn.Linear(QFE_OUT + LATENT_DIM, 128),

            nn.GELU(),

            nn.Dropout(0.08),

            nn.Linear(128, 64),

            nn.GELU(),

            nn.Linear(64, 10)
        )

    def forward(self, x):

        feat = self.cnn(x)

        latent = self.reducer(feat)

        latent_pi = torch.sigmoid(latent) * np.pi

        q_features = _QFE.extract(latent_pi)

        # IMPORTANT FOR SMALL DATA
        q_features = torch.tanh(q_features)

        hybrid = torch.cat(
            [latent, q_features],
            dim=1
        )

        return self.classifier(hybrid)

# =============================================================================
# CLASSICAL MODEL
# =============================================================================

class ClassicalCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.net = nn.Sequential(

            conv_block(1, 16, True),

            conv_block(16, 32, True),

            conv_block(32, 64, False),

            nn.Flatten(),

            nn.Linear(64 * 2 * 2, 128),

            nn.GELU(),

            nn.Dropout(0.10),

            nn.Linear(128, 10)
        )

    def forward(self, x):

        return self.net(x)

# =============================================================================
# DATA
# =============================================================================

digits = load_digits()

X_raw = (digits.images / 16.0).astype(np.float32)

y_raw = digits.target.astype(np.int64)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_raw,
    y_raw,
    train_size=1200,
    test_size=300,
    stratify=y_raw,
    random_state=42
)

def make_loader(X, y, shuffle=True):

    X = torch.tensor(X[:, None])

    y = torch.tensor(y)

    ds = TensorDataset(X, y)

    return DataLoader(
        ds,
        batch_size=128,
        shuffle=shuffle,
        pin_memory=True
    )

test_loader = make_loader(
    X_te,
    y_te,
    shuffle=False
)

# =============================================================================
# TRAIN FUNCTION
# =============================================================================

def train_model(model, train_loader, test_loader, epochs=10):

    model = model.to(device)

    model = torch.compile(model)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.AdamW(
        model.parameters(),
        lr=4e-4,
        weight_decay=2e-4
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs
    )

    scaler = torch.cuda.amp.GradScaler()

    history = {
        "train_acc": [],
        "val_acc": []
    }

    for ep in range(epochs):

        # =========================================================================
        # TRAIN
        # =========================================================================

        model.train()

        tc = tn = 0

        for xb, yb in train_loader:

            xb = xb.to(device)

            yb = yb.to(device)

            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast():

                out = model(xb)

                loss = criterion(out, yb)

            scaler.scale(loss).backward()

            scaler.step(optimizer)

            scaler.update()

            tc += (
                out.argmax(1) == yb
            ).sum().item()

            tn += len(xb)

        scheduler.step()

        train_acc = tc / tn

        # =========================================================================
        # VALIDATION
        # =========================================================================

        model.eval()

        vc = vn = 0

        preds = []

        labels = []

        with torch.no_grad():

            for xb, yb in test_loader:

                xb = xb.to(device)

                yb = yb.to(device)

                with torch.cuda.amp.autocast():

                    out = model(xb)

                vc += (
                    out.argmax(1) == yb
                ).sum().item()

                vn += len(xb)

                preds.extend(
                    out.argmax(1).cpu().numpy()
                )

                labels.extend(
                    yb.cpu().numpy()
                )

        val_acc = vc / vn

        history["train_acc"].append(train_acc)

        history["val_acc"].append(val_acc)

        print(
            f"Epoch {ep+1:02d} | "
            f"Train {train_acc:.4f} | "
            f"Val {val_acc:.4f}"
        )

    return history, np.array(preds), np.array(labels)

# =============================================================================
# EXPERIMENT
# =============================================================================

subset_sizes = [20, 100, 200, 400, 800]

quantum_accs = []

classical_accs = []

for size in subset_sizes:

    print("\n" + "=" * 70)

    print(f"TRAINING WITH {size} SAMPLES")

    print("=" * 70)

    X_small = X_tr[:size]

    y_small = y_tr[:size]

    train_loader = make_loader(
        X_small,
        y_small,
        shuffle=True
    )

    # -------------------------------------------------------------------------
    # QUANTUM
    # -------------------------------------------------------------------------

    print("\nHYBRID QUANTUM CNN")

    q_model = HybridQuantumCNN()

    q_hist, _, _ = train_model(
        q_model,
        train_loader,
        test_loader,
        epochs=10
    )

    q_acc = q_hist["val_acc"][-1] * 100

    quantum_accs.append(q_acc)

    print(f"QUANTUM ACCURACY : {q_acc:.2f}%")

    # -------------------------------------------------------------------------
    # CLASSICAL
    # -------------------------------------------------------------------------

    print("\nCLASSICAL CNN")

    c_model = ClassicalCNN()

    c_hist, _, _ = train_model(
        c_model,
        train_loader,
        test_loader,
        epochs=10
    )

    c_acc = c_hist["val_acc"][-1] * 100

    classical_accs.append(c_acc)

    print(f"CLASSICAL ACCURACY : {c_acc:.2f}%")

# =============================================================================
# ACCURACY GRAPH
# =============================================================================

plt.figure(figsize=(10, 6))

plt.plot(
    subset_sizes,
    quantum_accs,
    marker='o',
    linewidth=3,
    markersize=10,
    label="Hybrid Quantum CNN"
)

plt.plot(
    subset_sizes,
    classical_accs,
    marker='s',
    linewidth=3,
    markersize=10,
    label="Classical CNN"
)

plt.xlabel("Training Dataset Size")

plt.ylabel("Test Accuracy (%)")

plt.title("Quantum vs Classical Learning")

plt.grid(True)

plt.legend()

comparison_path = f"{OUT}/quantum_vs_classical.png"

plt.savefig(
    comparison_path,
    dpi=250,
    bbox_inches="tight"
)

print(f"\nSaved : {comparison_path}")

# =============================================================================
# FINAL TRAINING
# =============================================================================

print("\nFINAL QUANTUM TRAINING")

final_loader = make_loader(
    X_tr,
    y_tr,
    shuffle=True
)

final_model = HybridQuantumCNN()

history, preds, labels = train_model(
    final_model,
    final_loader,
    test_loader,
    epochs=14
)

# =============================================================================
# FINAL ACCURACY
# =============================================================================

acc = (preds == labels).mean() * 100

print("\nFINAL TEST ACCURACY")

print(f"{acc:.2f}%")

# =============================================================================
# CONFUSION MATRIX
# =============================================================================

cm = confusion_matrix(labels, preds)

plt.figure(figsize=(8, 7))

plt.imshow(cm)

plt.title(
    f"Confusion Matrix | Accuracy = {acc:.2f}%"
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

cm_path = f"{OUT}/confusion_matrix.png"

plt.savefig(
    cm_path,
    dpi=250,
    bbox_inches="tight"
)

print(f"Saved : {cm_path}")

# =============================================================================
# TRAINING CURVES
# =============================================================================

plt.figure(figsize=(10, 5))

plt.plot(history["train_acc"], linewidth=3)

plt.plot(history["val_acc"], linewidth=3)

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

plt.title("Final Quantum Training")

plt.legend(["Train", "Validation"])

plt.grid(True)

curve_path = f"{OUT}/training_curves.png"

plt.savefig(
    curve_path,
    dpi=250,
    bbox_inches="tight"
)

print(f"Saved : {curve_path}")

# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 70)

print("FINAL RESULTS")

print("=" * 70)

for s, q, c in zip(
    subset_sizes,
    quantum_accs,
    classical_accs
):

    print(
        f"Samples {s:4d} | "
        f"Quantum {q:6.2f}% | "
        f"Classical {c:6.2f}%"
    )

print("\nOUTPUT DIRECTORY")

print(OUT)

print("\nDONE.")


DEVICE : cpu
Quantum Expansion : 16 → 64

TRAINING WITH 20 SAMPLES

HYBRID QUANTUM CNN
Epoch 01 | Train 0.0000 | Val 0.1000
Epoch 02 | Train 0.0500 | Val 0.1000
Epoch 03 | Train 0.1000 | Val 0.1000
Epoch 04 | Train 0.1000 | Val 0.1000
Epoch 05 | Train 0.1000 | Val 0.1000
Epoch 06 | Train 0.1000 | Val 0.1000
Epoch 07 | Train 0.1000 | Val 0.1000
Epoch 08 | Train 0.1000 | Val 0.1000
Epoch 09 | Train 0.1000 | Val 0.1000
Epoch 10 | Train 0.1500 | Val 0.1000
QUANTUM ACCURACY : 10.00%

CLASSICAL CNN
Epoch 01 | Train 0.1500 | Val 0.1000
Epoch 02 | Train 0.1500 | Val 0.1000
Epoch 03 | Train 0.1500 | Val 0.1000
Epoch 04 | Train 0.1500 | Val 0.1000
Epoch 05 | Train 0.1500 | Val 0.1000
Epoch 06 | Train 0.1500 | Val 0.1000
Epoch 07 | Train 0.1500 | Val 0.1000
Epoch 08 | Train 0.1500 | Val 0.1000
Epoch 09 | Train 0.1500 | Val 0.1000
Epoch 10 | Train 0.1500 | Val 0.1000
CLASSICAL ACCURACY : 10.00%

TRAINING WITH 100 SAMPLES

HYBRID QUANTUM CNN
Epoch 01 | Train 0.0700 | Val 0.1000
Epoch 02 | Train 0.

In [ ]:
!pip install qiskit